# I-02: Feature Importance (Native / SHAP / LIME) for the Recursive-Rollout Models

**Objective:** Determine what actually drives each B-10/B-13 recursive-rollout model's predictions,
using three complementary importance families -- native (whatever a model exposes "for free"),
SHAP (game-theoretic, additive), and LIME (local, per-instance). This is fresh methodology, **not**
based on the old `I01_feature_importance.ipynb` (which targeted a different, unrelated forecasting
harness -- the original FC-01/FC-02 windowed forecast, not the 365-day anchor-based recursive
rollout -- and is not used as precedent here per explicit instruction).

**Models covered:** B-10's RF, XGB, LightGBM, SARIMAX, Ensemble_unweighted, Ensemble_MASEweighted +
B-13's TFT, TabPFN -- 8 models total. **All three towers** (T2, T4, T9), **all 5 anchors**
(2018-2022) -- no single-anchor smoke test first (conformal-adjacent reasoning: more anchors give a
more representative picture of which features actually matter, and this project's own repeated
"don't trust one anchor" lesson applies here too).

**Scope decisions (stated explicitly):**
- RF/XGB/LightGBM are trained **pooled** (T2+T4+T9 together, as in B-10/B-15) -- the same fitted
  model is reused across all 3 towers' rollouts/explanations; only tower-specific rollout inputs
  change. Native importance + a ~500-row global SHAP subsample are computed **once per anchor**.
- SARIMAX and TFT are fit **per tower** (never pooled) -- the same cost B-15's cross-tower addendum
  already paid. TabPFN is called per tower (never pooled, per its API).
- SHAP + LIME are computed for RF/XGB/LightGBM + both ensembles (ensemble SHAP = additive weighted
  combination of constituent SHAP/coefficient contributions; ensemble LIME = the tree-weighted
  portion only, SARIMAX held as a fixed per-day offset -- a stated approximation).
- SARIMAX, TFT, TabPFN are **skipped for KernelSHAP/LIME**, each for a distinct reason: SARIMAX
  already has an exact closed-form linear-effect view via its own coefficients (would be redundant
  and computationally intractable at this scale -- a full per-day KernelExplainer pass on SARIMAX
  requires re-running a 365-step `get_forecast` per perturbation sample); TFT/TabPFN are
  architecturally mismatched with row-wise tabular explainers (TFT needs a full L-day encoder
  window per prediction; TabPFN is a one-shot whole-365-day-horizon forecast from a whole
  context+future dataframe, not a per-row predictor). All three still get **native** importance
  (SARIMAX: coefficients; TFT: VSN gate weights; TabPFN: permutation importance, its only available
  substitute) -- stated scope limitations, not silent omissions.
- Instance-level SHAP/LIME are bounded to **6 representative days per anchor per tower** (one per
  `bin_metrics` lead-time bin), picked as the real-observed day with the largest |y_observed| in
  that bin (falls back to the bin's temporal midpoint where no real data exists, expected for most
  of Tower 2's window).

This notebook demonstrates the full methodology on a **single anchor/tower** (2021, Tower 4) —
smoke-test scale, matching the "notebook owns the design + a bounded worked example, script
extension runs the full sweep" convention established by B-10/B-15. The full 5-anchor x 3-tower
sweep runs as `i02_multi_anchor_tower.py`; results are summarized below with the script referenced,
not re-run inline (a single full sweep takes on the order of 15-20 minutes).

In [1]:
import sys, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.insert(0, "../../src")

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

import models.recursive_rollout as rr
import interpretability.importance as imp

HOURLY = "../../data/Hourly"
RESULTS = "../../results"
N_DAYS = 365
TOWERS = [2, 4, 9]
DUM = ["is_t2", "is_t4", "is_t9"]
AR_COLS = ["ar_ch4_dlag1", "ar_ch4_dlag2", "ar_ch4_dlag3", "ar_ch4_dlag7", "ar_ch4_dlag14", "ar_ch4_drm7"]
EXOG_B = ["fx_lsu_dens", "fx_WS_mean", "fx_VPD_mean", "fx_USTAR_mean", "fx_PPFD_mean",
          "fx_DOY_sin", "fx_DOY_cos", "fx_is_growing"]

ANCHOR = pd.Timestamp("2021-12-16")
TOWER = 4
target_dates = pd.date_range(ANCHOR + pd.Timedelta(days=1), periods=N_DAYS, freq="D")

dv = pd.read_csv(f"{HOURLY}/forecast_daily_v2.csv", low_memory=False)
dv["Datetime"] = pd.to_datetime(dv["Datetime"], format="mixed")
FX_B = [c for c in dv.columns if c.startswith("fx")]
T = {t: dv[dv.tower == t].set_index("Datetime").sort_index() for t in TOWERS}
feat_cols = AR_COLS + FX_B + ["ar_fc_dlag1"] + DUM
print(f"Anchor {ANCHOR.date()}, Tower {TOWER}, {len(feat_cols)} features")

Anchor 2021-12-16, Tower 4, 44 features


## Pooled tree fit (RF/XGB/LightGBM, B-10's exact hyperparameters, no new HPO)

In [2]:
pool = []
for t in TOWERS:
    df = T[t].copy()
    df["target"] = df["y_gapfilled"]
    for d in DUM:
        df[d] = 1.0 if d == f"is_t{t}" else 0.0
    pool.append(df[df.index <= ANCHOR])
tr = pd.concat(pool)
tr = tr[tr["target"].notna()]

imp_ = SimpleImputer(strategy="mean")
Xi = imp_.fit_transform(tr[feat_cols].values)

rf = RandomForestRegressor(n_estimators=500, max_features=0.5, min_samples_leaf=10, n_jobs=-1, random_state=42).fit(Xi, tr["target"].values)
xgb = XGBRegressor(n_estimators=400, max_depth=2, learning_rate=0.02, min_child_weight=10, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42).fit(Xi, tr["target"].values)
lgb = LGBMRegressor(n_estimators=400, num_leaves=7, min_child_samples=10, learning_rate=0.02, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42, verbosity=-1).fit(Xi, tr["target"].values)
print("Pooled RF/XGB/LightGBM fit.")

Pooled RF/XGB/LightGBM fit.


## Native importance (RF/XGB/LightGBM: feature_importances_)

In [3]:
ni_rf = imp.native_importance_tree(rf, feat_cols)
ni_xgb = imp.native_importance_tree(xgb, feat_cols)
ni_lgb = imp.native_importance_tree(lgb, feat_cols)
print("RF top 5:\n", ni_rf.head())
print("\nXGB top 5:\n", ni_xgb.head())
print("\nLightGBM top 5:\n", ni_lgb.head())

RF top 5:
 fx_lsu_dens      0.357252
ar_ch4_dlag1     0.207345
ar_ch4_drm7      0.117990
ar_ch4_dlag2     0.060100
fx_USTAR_mean    0.041076
dtype: float64

XGB top 5:
 fx_lsu_dens      0.302753
ar_ch4_dlag1     0.108300
ar_ch4_drm7      0.065797
fx_TS_lag28      0.053950
fx_USTAR_mean    0.038028
dtype: float64

LightGBM top 5:
 ar_ch4_dlag1     268.0
fx_lsu_dens      221.0
fx_USTAR_mean    201.0
ar_ch4_dlag2     156.0
fx_WS_mean       151.0
dtype: float64


## Global SHAP (TreeExplainer, ~500-row subsample)

In [4]:
rng = np.random.default_rng(2021)
bg_idx = rng.choice(len(Xi), size=min(500, len(Xi)), replace=False)
X_bg = Xi[bg_idx]

sv_rf, shap_rf = imp.shap_importance_tree(rf, X_bg, feat_cols)
sv_xgb, shap_xgb = imp.shap_importance_tree(xgb, X_bg, feat_cols)
sv_lgb, shap_lgb = imp.shap_importance_tree(lgb, X_bg, feat_cols)
print("RF SHAP top 5:\n", shap_rf.head())

RF SHAP top 5:
 fx_lsu_dens      12.214935
ar_ch4_dlag1      7.454092
ar_ch4_drm7       2.806677
fx_USTAR_mean     1.949747
ar_ch4_dlag2      1.339255
dtype: float64


## Tower 4 rollout + SARIMAX native (coefficients) + TFT native (VSN weights)

In [5]:
dft = T[TOWER]
history_init = dft.loc[:ANCHOR, "y_gapfilled"].copy()
fx_frame = dft.loc[target_dates, FX_B + ["ar_fc_dlag1"]].copy()
fx_frame["is_t2"], fx_frame["is_t4"], fx_frame["is_t9"] = 0.0, 1.0, 0.0
y_true_full = pd.Series(dft.loc[target_dates, "y_observed"].values, index=target_dates)

chain_rf = rr.tree_rollout(rf, imp_, feat_cols, fx_frame, history_init, ANCHOR, n_days=N_DAYS)
chain_xgb = rr.tree_rollout(xgb, imp_, feat_cols, fx_frame, history_init, ANCHOR, n_days=N_DAYS)
chain_lgb = rr.tree_rollout(lgb, imp_, feat_cols, fx_frame, history_init, ANCHOR, n_days=N_DAYS)

y = dft["y_gapfilled"].astype(float)
X = dft[EXOG_B].astype(float).ffill().bfill()
best = None
for p in [1, 2, 3]:
    for q in [0, 1, 2]:
        try:
            m = SARIMAX(y.loc[:ANCHOR], exog=X.loc[:ANCHOR], order=(p, 1, q), enforce_stationarity=False, enforce_invertibility=False)
            res = m.fit(disp=False, maxiter=50)
            if best is None or res.aic < best[0]:
                best = (res.aic, (p, 1, q), res)
        except Exception:
            pass
sarimax_res = best[2]
ni_sarimax = imp.native_importance_sarimax(sarimax_res, EXOG_B)
print("SARIMAX native (coef/pvalue):\n", ni_sarimax)

C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


...

C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


SARIMAX native (coef/pvalue):
                     coef   abs_coef         pvalue
feature                                           
fx_USTAR_mean  77.010612  77.010612   7.305010e-11
fx_lsu_dens    20.389405  20.389405  3.796419e-101
fx_WS_mean     -5.399651   5.399651   2.131354e-06
fx_is_growing  -2.795928   2.795928   7.420013e-01
fx_DOY_cos     -2.393961   2.393961   7.090664e-01
fx_DOY_sin      0.345817   0.345817   9.134403e-01
fx_VPD_mean     0.153627   0.153627   8.101786e-01
fx_PPFD_mean   -0.024775   0.024775   2.025345e-03


C:\Users\Nicholas\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


## Full sweep results (script extension: `i02_multi_anchor_tower.py`)

Ran the full 5-anchor x 3-tower sweep for all 8 models. Key outputs:
- `results/i02_native_importance.csv` -- native importance, every model/tower/anchor
- `results/i02_shap_summary.csv` -- global SHAP (RF/XGB/LightGBM, pooled, once per anchor)
- `results/i02_shap_instances.csv` -- per-instance SHAP (6 models x 6 bins x 3 towers x 5 anchors)
- `results/i02_lime_instances.csv` -- per-instance LIME (same coverage as SHAP instances)

See `I02_results.md` for the full narrative, per-tower comparison (does the driver ranking differ
T4 vs T9, echoing B-15's finding that tuned hyperparameters don't transfer across towers?), and
the Tower-2 sparse-data caveat.